# 07 · Repeat of the main protocol with fully disjoint validation (nested selection)

**Purpose.** Table I of the paper selects GA/PSO hyperparameters on the last 20 % of the training part, and the same block
also drives early stopping in final training. This notebook repeats the main SIDO protocol with **disjoint blocks**:

| Block (of the training part) | Used for |
|---|---|
| A: first 60 % | fitting candidates during the search |
| B: next 20 % | **scoring candidates / selecting hyperparameters** |
| C: last 20 % | early stopping in final training only (`validation_split=0.2` in `models.train_predict`) |

The search follows Algorithms 1–2 exactly (population/swarm 20 × 15 = **300 candidate positions**, 20 fitness epochs,
blend crossover, constriction PSO); only the fitness block changes (`search.fitness(mode='nested')`).

**What is re-run, and why not all eight models.** The default LSTM, GRU, BiLSTM, TCN, Transformer and MLP use fixed default
hyperparameters, so no validation block is used to select them, and their Table I results do not change. Only the
GA- and PSO-tuned LSTMs depend on the selection block. The default LSTM is re-run as a control that should reproduce Table I (≈2.16 %).

**Modes** (set `MODE` below)

| MODE | Searches | Final training | Mirrors | Estimated time (CPU) |
|---|---|---|---|---|
| `QUICK` | 1 GA + 1 PSO (seed 42) | 30 seeds per config | Table I | about 4–7 h, one commit |
| `FULL` | 5 GA + 5 PSO (seeds 42, 1042, …) | 5 seeds per config | Table V, rows "Main, 300" | about 12–18 h, 2–3 commits |

Times are estimates from the run times in the README, not measurements. `FULL` is the interpretable comparison, because a single
search varies a lot (1.6–2.6 % across the five shared-validation searches in Table V). Every expensive step is cached; the
notebook stops cleanly before Kaggle's 12-hour limit. To continue, *Add Input → Your Work* with the previous output and commit again.

**Kaggle settings:** Accelerator *None* (CPU) is fine; turn *Internet on*; *Save Version → Save & Run All (Commit)*.

In [ ]:
import os, sys, glob, time, pickle, shutil, subprocess, warnings
warnings.filterwarnings('ignore')
T0 = time.time()

# ------------------------------- CONFIGURATION -------------------------------
MODE = 'QUICK'                      # 'QUICK' or 'FULL' (see the table above)
TIME_LIMIT_H = 11.0                 # stop starting new tasks after this many hours (Kaggle limit is 12 h)
REPO_URL = 'https://github.com/iuttriutami/LSTM-GA-PSO-Stock-Forecasting-Benchmark.git'
# -----------------------------------------------------------------------------
N_SEARCH = 1 if MODE == 'QUICK' else 5
SEEDS_PER_CONFIG = 30 if MODE == 'QUICK' else 5
SEARCH_SEEDS = [42 + 1000 * k for k in range(N_SEARCH)]      # 42, 1042, ... as in notebooks 04a/04b
OUT = '/kaggle/working' if os.path.isdir('/kaggle/working') else './out_nested'
os.makedirs(OUT, exist_ok=True)

def find_repo():
    pats = ['/kaggle/input/**/src/benchmark/__init__.py', './src/benchmark/__init__.py', '../src/benchmark/__init__.py']
    for pat in pats:
        for p in glob.glob(pat, recursive=True):
            return os.path.abspath(os.path.join(os.path.dirname(p), '..', '..'))
REPO = find_repo()
if REPO is None:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, f'{OUT}/repo'], check=True)
    REPO = f'{OUT}/repo'
sys.path.insert(0, os.path.join(REPO, 'src'))
print('repository:', REPO, '| mode:', MODE, '| searches per optimizer:', N_SEARCH, '| seeds per config:', SEEDS_PER_CONFIG)

In [ ]:
import numpy as np, pandas as pd, matplotlib as mpl, matplotlib.pyplot as plt
from scipy import stats
from benchmark import data as D, models as M, search as S, evaluation as E, baselines as B
mpl.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 8, 'axes.spines.top': False, 'axes.spines.right': False})

def get_sido():
    cache = f'{OUT}/SIDO_features.csv'
    if os.path.exists(cache):
        return pd.read_csv(cache, parse_dates=['Date'])
    try:
        d = D.load_stock('SIDO.JK')
    except Exception as ex:                       # Yahoo blocked: use an uploaded CSV with the 16 paper features
        print('download failed:', repr(ex)[:120])
        d = None
        for f in glob.glob('/kaggle/input/**/*SIDO*.csv', recursive=True):
            t = pd.read_csv(f, parse_dates=['Date'])
            if all(c in t.columns for c in D.FEATURES) and 1400 <= len(t) <= 1460:
                d = t; print('using', f); break
        if d is None:
            raise RuntimeError('No SIDO data. Turn Internet on, or add a dataset with SIDO_features.csv '
                               '(made by scripts/download_data.py).')
    d.to_csv(cache, index=False)
    return d

df = get_sido()
y_all = df['Close'].values
n_tr = int(len(df) * M.TRAIN_FRAC)
rw_mape = E.mape(y_all[n_tr:], B.random_walk(y_all))
print(f'rows: {len(df)} (paper: 1,430) | random-walk one-step MAPE: {rw_mape:.3f}% (paper: 1.368%)')

In [ ]:
CACHE = f'{OUT}/cache_nested'; os.makedirs(CACHE, exist_ok=True)
for f in glob.glob('/kaggle/input/**/cache_nested/*.pkl', recursive=True):      # resume from a previous commit
    t = os.path.join(CACHE, os.path.basename(f))
    if not os.path.exists(t): shutil.copy(f, t)

elapsed = lambda: (time.time() - T0) / 3600
have = lambda key: os.path.exists(f'{CACHE}/{key}.pkl')
def cached(key, fn):
    p = f'{CACHE}/{key}.pkl'
    if os.path.exists(p):
        return pickle.load(open(p, 'rb'))
    r = fn()
    pickle.dump(r, open(p + '.tmp', 'wb')); os.replace(p + '.tmp', p)
    return r
STOPPED = False
print('cached tasks restored:', len(glob.glob(CACHE + '/*.pkl')))

In [ ]:
# ---- Step 1. Searches with the paper protocol (300 candidate positions) and NESTED fitness ----
lb, ub = S.bounds('original')
NEED_SEARCH_H = 2.5                  # conservative allowance for one search
def run_search(opt, seed):
    F = S.Evaluator(lambda hp: S.fitness(hp, df, D.FEATURES, mode='nested'))
    (S.ga_paper if opt == 'GA' else S.pso_paper)(F, seed, lb, ub)
    hp, val = F.best_at(300)
    return dict(hist=F.history, calls=F.calls, unique=len(F.memo), hp=hp, val=val)

SEARCH = {}
for r, sd in enumerate(SEARCH_SEEDS):
    for opt in ('GA', 'PSO'):
        key = f'search_{opt}_r{r}'
        if not have(key) and elapsed() + NEED_SEARCH_H > TIME_LIMIT_H:
            STOPPED = True; print('time guard: not starting', key); continue
        SEARCH[(opt, r)] = cached(key, lambda: run_search(opt, sd))
        s = SEARCH[(opt, r)]
        print(f"{opt} r{r}: calls {s['calls']}, unique {s['unique']}, best nested-validation MSE {s['val']:.5f}, "
              f"on bound: {S.on_bounds(s['hp'])} | {s['hp']} | {elapsed():.2f} h")

In [ ]:
# ---- Step 2. Final training of the selected configurations (early stopping on block C) ----
NEED_FINAL_H = 1.5 if SEEDS_PER_CONFIG >= 30 else 0.4
def run_final(hp, seeds):
    outs = [M.train_predict(hp, s, df, D.FEATURES) for s in seeds]
    y, rw = outs[0]['y'], outs[0]['rw']
    P = np.array([o['pred'] for o in outs])
    return dict(y=y, rw=rw, P=P, mape=[E.mape(y, p) for p in P], hp=hp)

seeds = [42 + k for k in range(SEEDS_PER_CONFIG)]
TASKS = [('LSTM (standard), control', 'control', M.DEFAULT_HP, list(range(42, 72)))]
TASKS += [(f'LSTM-{o} nested r{r}', f'final_{o}_r{r}', SEARCH[(o, r)]['hp'], seeds) for r in range(N_SEARCH) for o in ('GA', 'PSO') if (o, r) in SEARCH]
FINAL = {}
for name, key, hp, sd in TASKS:
    if not have(key) and elapsed() + NEED_FINAL_H > TIME_LIMIT_H:
        STOPPED = True; print('time guard: not starting', name); continue
    FINAL[name] = cached(key, lambda: run_final(hp, sd))
    print(f"{name}: mean MAPE {np.mean(FINAL[name]['mape']):.3f}% (n={len(sd)}) | {elapsed():.2f} h")
if STOPPED:
    print('\nSTOPPED EARLY: add this notebook\'s output as input (Add Input -> Your Work) and commit again to continue.')

In [ ]:
# ---- Step 3. Statistics: DM versus the random walk (Holm-adjusted) and comparison with shared validation ----
rows = []
for name, f in FINAL.items():
    y, rw, P = f['y'], f['rw'], f['P']
    dm, p = E.dm_test(y - P.mean(0), y - rw)              # seed-averaged forecast, DM > 0: random walk more accurate
    m = np.array(f['mape']); ytr = y_all[:n_tr]
    lo, hi = E.ci95(m) if len(m) > 1 else (np.nan, np.nan)
    o = name.split()[0].replace('LSTM-', '')
    r = int(name.split(' r')[-1]) if ' r' in name and 'nested' in name else -1
    s = SEARCH.get((o, r))
    rows.append(dict(model=name, n_seeds=len(m), MAPE_mean=m.mean(), MAPE_sd=m.std(ddof=1) if len(m) > 1 else np.nan,
                     CI_lo=lo, CI_hi=hi, MASE_mean=np.mean([E.mase(y, p_, ytr) for p_ in P]), DM_vs_RW=dm, p_raw=p,
                     unique_configs=s['unique'] if s else np.nan, nested_val_MSE=s['val'] if s else np.nan,
                     on_bound=S.on_bounds(s['hp']) if s else '', **({f'hp_{k}': v for k, v in f['hp'].items()})))
R = pd.DataFrame(rows)
R['p_holm'] = E.holm(R.p_raw.values) if len(R) else []
R['RW_better_sig'] = (R.DM_vs_RW > 0) & (R.p_holm < 0.05)
R['MAPE_RW'] = rw_mape
R.round(4).to_csv(f'{OUT}/TableI_nested_repeat_runs.csv', index=False)
print(R[['model', 'n_seeds', 'MAPE_mean', 'MAPE_sd', 'MASE_mean', 'DM_vs_RW', 'p_holm', 'RW_better_sig', 'on_bound']].round(3).to_string(index=False))

# comparison with the shared-validation results shipped in the repository
res = os.path.join(REPO, 'results'); SUM = []
t1 = pd.read_csv(f'{res}/TableI_one_step.csv') if os.path.exists(f'{res}/TableI_one_step.csv') else None
t5 = pd.read_csv(f'{res}/TableV_E7a_paper_protocol_runs.csv') if os.path.exists(f'{res}/TableV_E7a_paper_protocol_runs.csv') else None
for o in ('GA', 'PSO'):
    nr = R[R.model.str.startswith(f'LSTM-{o} nested')]
    if nr.empty: continue
    d = dict(optimizer=o, searches=len(nr), nested_mean=nr.MAPE_mean.mean(), nested_sd_across_runs=nr.MAPE_mean.std(ddof=1) if len(nr) > 1 else np.nan,
             nested_min=nr.MAPE_mean.min(), nested_max=nr.MAPE_mean.max(), all_RW_better_sig=bool(nr.RW_better_sig.all()))
    if MODE == 'QUICK' and t1 is not None:                # Table I: single shared search, 30 seeds
        ref = t1[t1.Model == f'LSTM-{o}'].iloc[0]; a = nr.iloc[0]
        t, pv = stats.ttest_ind_from_stats(a.MAPE_mean, a.MAPE_sd, a.n_seeds, ref.MAPE, ref.SD, ref.n, equal_var=False)
        d.update(shared_mean=ref.MAPE, shared_sd=ref.SD, welch_p_nested_vs_shared=pv)
    if MODE == 'FULL' and t5 is not None:                 # Table V "Main, 300": five shared searches
        sh = t5[(t5.optimizer == o) & (t5.budget == 300)].test_MAPE_mean.values
        d.update(shared_mean=sh.mean(), shared_sd_across_runs=sh.std(ddof=1), shared_min=sh.min(), shared_max=sh.max(),
                 mannwhitney_p_nested_vs_shared=stats.mannwhitneyu(nr.MAPE_mean.values, sh, alternative='two-sided').pvalue)
    SUM.append(d)
SUMDF = pd.DataFrame(SUM); SUMDF.round(4).to_csv(f'{OUT}/TableI_nested_repeat_summary.csv', index=False)
print('\n', SUMDF.round(3).to_string(index=False))
ctl = R[R.model.str.contains('control')]
if len(ctl) and t1 is not None:
    print(f"\ncontrol (default LSTM): {ctl.MAPE_mean.iloc[0]:.3f}% vs Table I {t1[t1.Model == 'LSTM (standard)'].MAPE.iloc[0]:.3f}%  -> pipeline check")
H = pd.DataFrame([dict(optimizer=o, run=r, calls=c, distinct=dn, best_val_mse=v) for (o, r), s in SEARCH.items() for c, dn, v, _ in s['hist']])
if len(H): H.to_csv(f'{OUT}/SearchHistory_nested.csv', index=False)

In [ ]:
# ---- Step 4. Figure: per-seed MAPE of the nested-selected configurations ----
fig, ax = plt.subplots(figsize=(3.4, 2.8))
names = [n for n in FINAL if 'nested' in n]
for i, n in enumerate(names):
    ax.scatter(np.full(len(FINAL[n]['mape']), i) + np.random.RandomState(i).uniform(-.12, .12, len(FINAL[n]['mape'])),
               FINAL[n]['mape'], s=6, alpha=.6, color='#d62728' if 'PSO' in n else '#1f77b4')
    ax.plot([i - .25, i + .25], [np.mean(FINAL[n]['mape'])] * 2, color='black', lw=1.2)
ax.axhline(rw_mape, color='black', lw=1); ax.text(-.4, rw_mape - .04, f'Random walk ({rw_mape:.2f})', va='top', fontsize=7)
ax.set_xticks(range(len(names))); ax.set_xticklabels([n.replace(' nested ', '\n') for n in names], fontsize=6.5)
ax.set_ylabel('One-step test MAPE (%)'); ax.set_ylim(bottom=min(1.2, rw_mape - .2))
fig.tight_layout(); fig.savefig(f'{OUT}/Fig_nested_repeat.png', dpi=600, bbox_inches='tight'); plt.show()

print('\n=== READING THE RESULT ===')
if len(R):
    print('Every nested-selected configuration is significantly less accurate than the random walk (Holm-adjusted):',
          bool(R[R.model.str.contains('nested')].RW_better_sig.all()))
print('This repeats the main SIDO experiment with disjoint selection and early-stopping blocks. It does not change the five reference')
print('architectures (fixed defaults) and covers SIDO only; report it as a robustness result with these limits.')
print(f'Total time: {elapsed():.2f} h' + ('  (INCOMPLETE: see time-guard messages above)' if STOPPED else ''))